# Preprocess code 4 - B app
## Universidad ICESI 
### David Mauricio Orozco Rios
### author: Davoroz06 - IG

In [ ]:
# libraries
import pandas as pd
import numpy as np
import os
from datetime import datetime
from scipy.stats import gmean
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

# project paths and shared helpers (see src/config.py)
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "config.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from config import *  # wd, wd_dp, wd_db, wd_dpr, wd_re, wd_rp, wd_rt ...
from utils import homogenize_text


In [ ]:
data = pd.read_csv(wd_dp + "B_raw.csv")
len(data)

Join columns with same meaning

In [ ]:
data["descripcion"] = data['descripcion'].fillna(data['Descripcion'])
data["precio"] = data['precio'].fillna(data['Precio'])
data["precio_medida"] = data['precio_medida'].fillna(data['Precio_medida'])
data["fecha"] = data['Fecha'].fillna(data['dia'])
data["link"] = data['link'].fillna(data['Link'])
data["palabra"] = data['palabra'].fillna(data['item'])
data['palabra'] = data['palabra'].fillna(data['link'].str.extract(r"https?://[^/]+/search\?query=(.*)").squeeze())

Filter empty products

In [ ]:
data = data[~data['descripcion'].isna()]
len(data)

String homogenize

In [ ]:
data["palabra"] = data["palabra"].apply(homogenize_text)
data["descripcion"] = data["descripcion"].apply(homogenize_text)
data["tienda"] = data["tienda"].astype("str")
data["tienda"] = data["tienda"].apply(homogenize_text)

Price homogenize

In [ ]:
data["precio"] = data["precio"].astype("str")
data["precio"] = data["precio"].str.replace("\.0$", "", regex=True)
data["precio"] = data["precio"].str.replace("[a-zA-Z]", "", regex=True)
data["precio"] = data["precio"].str.replace(".", "")
data["precio"] = data["precio"].str.replace(",", ".")
data["precio"] = data["precio"].str.replace("$", "")

Filter empty price products

In [ ]:
data = data[data["precio"].str.len()!=0]
data = data[~data['precio'].isna()]

Price to numeric

In [ ]:
data["precio"] = data["precio"].astype("float")

In [ ]:
data.columns

Create size diferentiation for duplicated products

In [ ]:
data['precio_medida'] = data['precio_medida'].astype(str)
data['solo_numeros'] = data["precio_medida"].str.extract(r'([\d.]+)', expand=False)
data['solo_numeros'] = data['solo_numeros'].str.replace(r'(\.(0$|00$))','', regex = True)
data['solo_numeros'] = data['solo_numeros'].astype(float)
data['medida'] = data['precio']/data['solo_numeros']
data['medida'] = round(data['medida'], 1)
data['medida'] = data['medida'].astype(str)

In [ ]:
data = data[~data['tienda'].isna()]
data = data[data['tienda']!= 'nan']

In [ ]:
data['tienda'].unique()

In [ ]:
import hashlib
data['tienda'] = data['tienda'].fillna("")
# anonymize the stores listed inside the platform (short hash of the store name)
data['tienda'] = data['tienda'].apply(lambda x: "store_" + hashlib.sha1(x.encode("utf-8")).hexdigest()[:6])
data['medida'] = data['medida'].fillna("")
#data['descripcion'] = data['tienda'] + "_" + data['descripcion']
data['descripcion'] = data['tienda'] + "_" + data['medida'] + "_" + data['descripcion']

Select columns

In [ ]:
data["tienda"] = 'B'
data = data[["fecha", "descripcion", "precio", "palabra", "tienda"]]

Select unique products and define unique daily prices

In [ ]:
data = data.groupby(['fecha', 'descripcion', 'tienda']).agg({'precio': 'min'}).reset_index()

Show data

In [ ]:
data.head()

Save preprocess data

In [ ]:
pd.DataFrame(data.to_excel(wd_db+"B_clean.xlsx"))

In [ ]:
pd.DataFrame(data.to_csv(wd_db+"B_clean.csv"))